# 05.2 — Two read-only tools

[Class index and setup](README.md) · [Previous: 05.1](05.1-models-and-structured-answers.ipynb) · [Next: 05.3](05.3-a-bounded-harness.ipynb)

**Purpose:** Connect model proposals to two supplied, read-only Python functions.

**In class:** 30 minutes. **Independent work:** 15 minutes, included in the index's 210-minute allowance. Preparation has its own 30 minutes. The instructor's project is additional.

**Starting knowledge:** Basic Python functions, dictionaries, JSON/schema validation, and the earlier pytest ideas. All code and saved inputs are supplied again; no earlier exercise answers or paid key are needed.

**Learning outcomes:** Inspect a tool schema and call ID; distinguish unknown names from invalid arguments; complete one dictionary lookup.

[Glossary](GLOSSARY.md) · [Supplied code](report_app/__init__.py) · [Fixture provenance](fixtures/README.md)

In [ ]:
from pathlib import Path
import sys

if not Path("report_app/harness.py").is_file() or not Path("fixtures/world-bank-population.json").is_file():
    raise FileNotFoundError(
        "Open 05-small-ai-applications.code-workspace and restart the kernel. "
        "Download the complete class folder with report_app/, fixtures/, and solutions/."
    )
print("Notebook kernel:", sys.executable)
print("Class assets available. Default execution is REPLAY, with no HTTP or credentials.")

## In class · 30–38 minutes — Two functions, two saved countries

The dataset is the same small World Bank snapshot used in class 04. Each validated `Observation` has a nonempty `label` and a finite nonnegative `value`. `PRT:2023` identifies Portugal, `ESP:2023` Spain, and the unit is **persons**. These are saved 2023 estimates, not a current population service.

`get_summary()` returns count, total, and mean for the two observations. `find_observations(label)` returns exact matches as a list; a missing label returns `[]`. Neither tool writes files, sends HTTP, executes arbitrary code, or downloads a model. A tool's ordinary Python implementation is supplied and can be tested without an LLM.

In [ ]:
from report_app.source import load_observations
from report_app.tools import get_summary, find_observations

print([record.model_dump() for record in load_observations()])
print("Summary:", get_summary())
print("Portugal:", find_observations("PRT:2023"))
print("Missing France:", find_observations("FRA:2023"))
assert get_summary()["total"] == 58930702
assert find_observations("FRA:2023") == []

### A narrow contract is easier to inspect

The model sees descriptions and JSON Schemas for the two tool names. It proposes a name and JSON-encoded arguments; our dispatcher validates them. `get_summary` accepts an empty object. `find_observations` requires exactly one nonempty string field, `label`. Extra arguments and numeric labels are rejected. An unknown country label is still a valid string; the operation returns an empty result.

`strict=True` requests schema adherence from the provider, and Pydantic checks again in Python. We keep the application check because replay, changed code, and unexpected input still cross this boundary. Pydantic's strict argument model does not change the earlier Observation model's documented numeric coercion behavior.

In [ ]:
from report_app.tools import FindArguments, TOOL_DEFINITIONS

print("Lookup schema:", FindArguments.model_json_schema())
print("Offered tool names:", [tool["name"] for tool in TOOL_DEFINITIONS])

## In class · 38–50 minutes — Trace one proposed call

The saved proposal is a Responses `function_call`. Its `arguments` field is JSON text, not a Python dictionary yet. The `call_id` connects this proposal to the result returned to the next model request. It is not the tool's name.

Read this as a sequence: inspect proposal → validate arguments → select permitted function → execute Python → encode result. Construction/import does none of that network work. The same supplied functions run when no model is present.

In [ ]:
import json
import inspect
from report_app.source import read_fixture
from report_app.tools import dispatch

call = read_fixture("replays/portugal.json")["steps"][0]["response"]["output"][0]
print("REPLAY proposal:", call)
validated = FindArguments.model_validate_json(call["arguments"])
print("Validated:", validated.model_dump())
result = dispatch(call["name"], call["arguments"])
returned = {"type": "function_call_output", "call_id": call["call_id"], "output": json.dumps(result)}
print("Return to next request:", returned)
print(inspect.getsource(dispatch))

### Visual guide — Link the proposal and result

<img src="images/linked-tool-round-trip.png" alt="Request one returns a lookup proposal with call ID c1; Python validates and executes it, returns output with c1, and request two produces a final answer." width="650" style="max-width: 100%; height: auto; background-color: white;">

Use the same call_id for the proposed function call and its returned result. Arguments arrive as JSON text; Python validates them before execution. This sequence represents a successful replay or live flow, not a network action performed by the picture.

*Credit: Original ADPRO course diagram (2026), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). [Editable Mermaid source](images/linked-tool-round-trip.mmd); [sources and teaching notes](IMAGE_SUGGESTIONS.md).*

### Permission and validity are separate questions

The dictionary `PERMITTED` is the complete allowlist. Looking up `delete_data` raises `KeyError` before any function can run. Looking up a permitted name with a numeric label raises `ValidationError` before that function runs. `eval`, `exec`, dynamic imports, and model-written Python are never used.

These checks enforce policy even if a prompt asks to change records or a returned string asks to ignore the tool rules. Read-only behavior comes from the supplied operations and dispatch policy, not from trusting an instruction in model output.

In [ ]:
from pydantic import ValidationError

for name, arguments in [("delete_data", "{}"), ("find_observations", '{"label":123}')]:
    try:
        dispatch(name, arguments)
    except (KeyError, ValidationError) as exc:
        print(name, "→", type(exc).__name__, "before execution")

Copy this into your **answer cell**; the single blank is the whole coding task:

```python
allowed_tools = {"find_observations": find_observations}
name = "find_observations"
validated = FindArguments(label="PRT:2023")
function = allowed_tools[____]
result = function(**validated.model_dump())
assert result[0]["value"] == 10578174
```

The actual application dispatcher is already complete. You do not replace it or implement a validation layer.

### Exercise 05.2.E1 — Select the supplied function

> **In class · 10 minutes · 50–60**
>
> **Goal:** Complete one dictionary lookup, with validation already supplied.
>
> **Start here:** Copy the short snippet above; both the function and validated arguments already exist.
>
> **Task:** Replace only `____` with the dictionary key variable. Run it, then run the supplied unknown-name and invalid-argument examples above. Explain their different failures.
>
> **Check:** Return the Portugal list with value 10,578,174; the two invalid proposals must not execute a tool.
>
> **Hint:** Use `name`, exactly as in an ordinary dictionary lookup.

In [ ]:
# Your attempt for 05.2.E1.
# Complete only the tiny expression; record observations/explanations here.

In [ ]:
# Reveal only after attempting the exercise.
# Remove the leading '# ' from the next line and run this cell.
# %load solutions/exercise_05_2_e1.py

## Guided follow-up — Independent work · 15 minutes

Spend five minutes adapting your connected snippet: change only the validated label from `PRT:2023` to `FRA:2023`, run the same selected function, and check `result == []` instead of indexing a missing row. Record the changed result, then compare the shorter `PRT` label. Spend five tracing name → validated arguments → function → result. Spend five explaining why `[]` can be a valid answer from a tool while “unknown function name” is a policy failure. This contributes 15 minutes to the guided-walkthrough allocation.

**Completion:** show the connected lookup, your changed validated input and observed present/absent results, plus the rejected-name/invalid-argument checks. This is a scaffolded build from supplied components; the dispatcher and loop remain supplied. Explain that only Python executes the permitted function. No new query syntax or tool implementation is required.

## References

[Official function-calling guide](https://developers.openai.com/api/docs/guides/function-calling) · [Local tool definitions](report_app/tools.py)